# Agentic RAG Graph Visualizer + Test Notebook

This notebook helps you:
- visualize the LangGraph flow
- test each component (router, web search, agent, retriever, reranker, generator)
- run an end-to-end query through the full graph

## 1. Environment Setup

If you are on Google Colab, run this cell first.

## 2. Configure Keys

Set keys in the current notebook runtime.

In [ ]:
import os

In [ ]:
# Required for generation
os.environ.setdefault('GOOGLE_API_KEY', 'YOUR_GOOGLE_API_KEY')

# Required for embeddings
os.environ.setdefault('HF_API_KEY', 'YOUR_HF_API_KEY')

# Optional Langfuse tracing
os.environ.setdefault('LANGFUSE_PUBLIC_KEY', '')
os.environ.setdefault('LANGFUSE_SECRET_KEY', '')
os.environ.setdefault('LANGFUSE_BASE_URL', 'https://cloud.langfuse.com')

print('GOOGLE_API_KEY set:', bool(os.environ.get('GOOGLE_API_KEY') and os.environ['GOOGLE_API_KEY'] != 'YOUR_GOOGLE_API_KEY'))
print('HF_API_KEY set:', bool(os.environ.get('HF_API_KEY') and os.environ['HF_API_KEY'] != 'YOUR_HF_API_KEY'))
print('Langfuse enabled:', bool(os.environ.get('LANGFUSE_PUBLIC_KEY') and os.environ.get('LANGFUSE_SECRET_KEY')))

## 3. Imports

In [ ]:
from pprint import pprint

from app.agent.agent import build_agent, retriever_selector_tool, web_search_tool
from app.generation.llm import get_llm
from app.generation.rag_graph import build_graph, retrieve, rerank, generate
from app.pipeline.ingestion_pipeline import IngestionPipeline
from app.retrieval.router import select_retrieval_strategy
from app.retrieval.web_search import web_search

print('Imports OK')

## 4. Build and Visualize Graph

In [ ]:
from IPython.display import Image, Markdown, display

import importlib

import app.generation.rag_graph as rag_graph_module



importlib.reload(rag_graph_module)

graph = rag_graph_module.build_graph()

print('Graph compiled successfully')



# Try PNG render first

try:

    png_bytes = graph.get_graph().draw_mermaid_png()

    display(Image(png_bytes))

except Exception as e:

    print('PNG render unavailable, showing Mermaid text instead:', e)

    mermaid_text = graph.get_graph().draw_mermaid()

    display(Markdown('```mermaid\n' + mermaid_text + '\n```'))


## 5. Ingest Sample Data

Run this once so retriever/reranker has context to work with.

In [ ]:
from pathlib import Path



pipeline = IngestionPipeline()



candidates = ['data/sample.html', 'data/sample.csv', 'data/sample.pdf']

ingested = []

failed = []



for fp in candidates:

    if not Path(fp).exists():

        continue



    print('Ingesting:', fp)

    try:

        result = pipeline.run(fp)

        if result:

            ingested.append(fp)

        else:

            failed.append((fp, 'No chunks produced'))

    except Exception as e:

        failed.append((fp, str(e)))

        print('  -> skipped due to error:', e)



if not ingested:

    raise RuntimeError(f'No documents ingested successfully. Failures: {failed}')



print('Ingested files:', ingested)

if failed:

    print('Skipped files:')

    for item in failed:

        print(' -', item[0], '|', item[1])


## 6. Test Retrieval Strategy Router

In [ ]:
queries = [
    'Find exact keyword match for product code: ABC-123',
    'What does the document say about architecture?',
    'latest news about retrieval augmented generation',
    ''
]

for q in queries:
    strategy = select_retrieval_strategy(q)
    print(f'query={q!r} -> strategy={strategy}')

## 7. Test Web Search Tool

In [ ]:
web_results = web_search('What is LangGraph?', max_results=3)
print('Raw web_search results:')
pprint(web_results)

print('Tool wrapper output:')
print(web_search_tool.invoke({'query': 'What is LangGraph?'}))

## 8. Test Agent Only

In [ ]:
llm = get_llm()
agent = build_agent(llm)

agent_response = agent.invoke({
    'messages': [{'role': 'user', 'content': 'How should I retrieve for an exact code lookup?'}]
})

print('Agent response:')
pprint(agent_response)

## 9. Test Individual Graph Nodes

In [ ]:
state = {'query': 'Summarize the key points from the ingested sample files.'}

retrieved = retrieve(state)
print('retrieve -> docs:', len(retrieved.get('docs', [])))

reranked = rerank({**state, **retrieved})
print('rerank -> compressed_docs:', len(reranked.get('compressed_docs', [])))

generated = generate({**state, **retrieved, **reranked})
print('generate -> answer preview:')
print(generated['answer'][:500])

## 10. Full End-to-End Graph Test

In [ ]:
result = graph.invoke({'query': 'Give me a concise summary of the ingested documents.'})

print('Final answer:')
print(result.get('answer', 'No answer'))

print('---')
print('Agent output preview:')
print(str(result.get('agent_output', ''))[:500])

## 11. Optional: Retry Behavior Demo

This validates that retry wrapper attaches errors to state before retrying.

In [ ]:
from app.generation.retry_utils import retry_with_exception

calls = {'n': 0}

@retry_with_exception
def flaky(state):
    calls['n'] += 1
    if calls['n'] < 3:
        raise RuntimeError('simulated failure')
    return {'ok': True, 'attempts': calls['n'], 'error_in_state': state.get('error', '')}

demo_state = {}
print(flaky(demo_state))